<a href="https://colab.research.google.com/github/LaxmiPrasannaMaduri/Bda_assignment/blob/main/RecommendationEngine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("RecommendationEngine").getOrCreate()

In [ ]:
ratings_data = [
    (1, 101, 4.0),  # userId, movieId, rating
    (1, 102, 3.5),
    (2, 101, 5.0),
    (2, 103, 4.0),
    (3, 102, 2.0),
    (3, 103, 4.5),
]

columns = ["userId", "movieId", "rating"]
ratings_df = spark.createDataFrame(ratings_data, columns)

In [ ]:
train_df, test_df = ratings_df.randomSplit([0.5, 0.5], seed=42)

In [ ]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="nan",
    nonnegative=True,
    rank=10,
    maxIter=10,
    regParam=0.1
)

model = als.fit(train_df)

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test_df)

# Filter out NaN predictions that arise from cold start items if coldStartStrategy="nan"
# The evaluator cannot handle NaN values directly.
cleaned_predictions = predictions.filter(predictions["prediction"].isNotNull())

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

# Only evaluate if there are valid predictions
if cleaned_predictions.count() > 0:
    rmse = evaluator.evaluate(cleaned_predictions)
    print("Root-mean-square error (RMSE):", rmse)
else:
    print("No valid predictions to evaluate after filtering NaN values.")

Root-mean-square error (RMSE): nan


In [ ]:
# Top 2 movie recommendations for each user
user_recs = model.recommendForAllUsers(2)
user_recs.show(truncate=False)

# Top 2 user recommendations for each movie
movie_recs = model.recommendForAllItems(2)
movie_recs.show(truncate=False)

+------+-----------------+
|userId|recommendations  |
+------+-----------------+
|3     |[{103, 4.442769}]|
+------+-----------------+

+-------+---------------+
|movieId|recommendations|
+-------+---------------+
|103    |[{3, 4.442769}]|
+-------+---------------+

